In [ ]:
uv add tensorflow==2.19.0
uv add tensorflow.keras==3.13.2

In [ ]:

python -m venv .myenv
.\.myenv\Scripts\activate

pip install tensorflow==2.13.0
pip install ipykernel

In [2]:
import tensorflow as tf
tf.__version__
# print(hasattr(tf, "keras"))

'2.13.0'

In [3]:
from tensorflow import keras
print(keras.__file__)

e:\COLLEGE\WORK\AI ML\Projext\HorticultureHelpingHand\.myenv\lib\site-packages\keras\api\_v2\keras\__init__.py


In [ ]:
import os
import json
from zipfile import ZipFile
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Input, layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [ ]:

kaggle_cred=json.load(open("kaggle.json"))
os.environ['KAGGLE_USERNAME']= kaggle_cred['username']
os.environ['KAGGLE_KEY']= kaggle_cred['key']
!kaggle datasets download abdallahalidev/plantvillage-dataset
with ZipFile("plantvillage-dataset.zip", "r") as zip_file:
  zip_file.extractall()

In [ ]:
os.listdir("plantvillage dataset")
print(len(os.listdir("plantvillage dataset/color")))
os.listdir("plantvillage dataset/color")[:5]
# Dataset path
base_directory="plantvillage dataset/color"


In [ ]:


image_path="/content/plantvillage dataset/color/Apple___Apple_scab/0340dc35-5215-48ab-8db7-06af99fcb358___FREC_Scab 2966.JPG"


# Read the image
img= mpimg.imread(image_path)
print(img.shape)
plt.imshow(img)
plt.axis("off")
plt.show()


In [ ]:

# Hyperparameters
image_size = 224
batch_size = 32
base_directory = "plantvillage dataset/color"

# Augmentation in ImageDataGenerator (for training)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    base_directory,
    target_size=(image_size, image_size),
    batch_size=batch_size,
    subset="training",
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    base_directory,
    target_size=(image_size, image_size),
    batch_size=batch_size,
    subset="validation",
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False
)

In [ ]:


num_classes = train_generator.num_classes
print(f"Number of classes: {num_classes}")



In [ ]:


model = models.Sequential([
    Input(shape=(image_size, image_size, 3)),

    layers.Conv2D(32, (3,3), activation="relu", padding='same',kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3,3), activation="relu", padding='same',kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3,3), activation="relu", padding='same',kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3,3), activation="relu", padding='same',kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu",kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)



In [ ]:
model.summary()

In [ ]:

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
training = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=15,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
import matplotlib.pyplot as plt

# Plotting Training and Validation Accuracy
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(training.history['accuracy'], label='Train Accuracy')
plt.plot(training.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy During Training')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)


In [ ]:

# Plotting Training and Validation Loss
plt.subplot(1, 2, 2)
plt.plot(training.history['loss'], label='Train Loss')
plt.plot(training.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("Evaluating model...")
val_loss, val_acc=model.evaluate(validation_generator, steps=validation_generator.samples // batch_size)
print(f"Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
def load_preprocess_img(image_path, target_size=(224,224)):
  img=Image.open(image_path).convert('RGB')
  img=img.resize(target_size)
  img_arr=np.array(img)
  img_arr=img_arr/255.0 #rescale
  img_arr=np.expand_dims(img_arr, axis=0) # making it suitable as batch input
  # img_arr=img_arr.astype("float32")/255.
  return img_arr

def predict_img_class(model, image_path, class_indices):
  preprocessed_img=load_preprocess_img(image_path)
  predicted_class=model.predict(preprocessed_img)
  predicted_class_index=np.argmax(predicted_class, axis=1)[0]
  predicted_class_name=class_indices[predicted_class_index]
  return predicted_class_name


In [ ]:
class_indices={v:k for k, v in train_generator.class_indices.items()}

In [ ]:
# sample_image_path = "/content/plantvillage dataset/color/Strawberry___Leaf_scorch/01647a51-6ee5-4686-8f60-26ebed68fe21___RS_L.Scorch 0130.JPG"

sample_image_path = "/content/1.JPG" # newelaf2 -> strawberry leaf scorch
sample_image_path = "/content/Screenshot 2026-04-09 224912.png"
predicted_class = predict_img_class(model, sample_image_path, class_indices)
print(f"The predicted class for the sample image is: {predicted_class}")


In [ ]:
true_label = "Strawberry___Leaf_scorch"

# Load the image
img = mpimg.imread(sample_image_path)

# Display the image
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(f"True Label: {true_label}\nPredicted Label: {predicted_class}")
plt.axis("off")
plt.show()

print(f"True Label for {sample_image_path}: {true_label}")
print(f"Predicted Label for {sample_image_path}: {predicted_class}")

In [ ]:
# saving the class_indices as json
json.dump(class_indices, open("class_indices.json","w"))

model.save("fruits_disease_prediction_model_updated.h5")